# Analysis of the financial data

In this notebook we look at the financial data that came with the project. The optional task asks for a statistical analysis of the financial distributions across industries, and it says that we have to handle outliers and missing data. That is what we do here.

The file has four columns we can use: `net_profit_margin`, `ebitda_margin`, `asset_turnover` and `country_code`. The first three are ratios and country is a category, so country sometimes needs a different method. We say so when that happens.

We first check what is wrong with the data, then clean it, and only then compare the industries. Comparing them first would not make much sense, because a few extreme values would decide the result.

The notebook runs in under a minute, since no model is trained here.

One thing to keep in mind: we calculate percentiles and medians on all rows here, which is fine for describing the data. A model has to learn them from the training rows only, and `02_final_ensembles.ipynb` does it that way.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

DATA_DIR = "../../data/"   # folder with df_train.pkl and df_financials_train.pkl
RATIOS = ["net_profit_margin", "ebitda_margin", "asset_turnover"]
MIN_COMPANIES = 20        # smaller countries go into "other", same as min_frequency in 02_final_ensembles.ipynb

# 1. Loading Data

Same as in `02_final_ensembles.ipynb`, we first remove the duplicate embeddings, because the private test set is cleaned too.

162 ids appear twice in the financial file. Usually one row has the values and the other row only has the country. We checked all of them, and the two rows of a company never disagree. So `groupby("id").first()` is safe here: it takes the first non-empty value of each column.

In [2]:
df = pd.read_pickle(DATA_DIR + "df_train.pkl")
df = df.drop_duplicates(subset="business_description_embedding").reset_index(drop=True)

df_fin = pd.read_pickle(DATA_DIR + "df_financials_train.pkl").groupby("id").first()
data = df[["id", "industry"]].join(df_fin, on="id")
data["no_financial_row"] = ~data["id"].isin(df_fin.index)

print("companies:", len(data), "| industries:", data["industry"].nunique())
print("companies without a financial row:", data["no_financial_row"].sum())
data.head()

companies: 27128 | industries: 25
companies without a financial row: 2206


,id,industry,country_code,net_profit_margin,ebitda_margin,asset_turnover,no_financial_row
0,42576,Banks,USA,0.155503,NaN,0.033292,False
1,21752,Materials,DEU,0.023660,NaN,0.861915,False
2,804,Commercial & Professional Services,CHN,0.001364,NaN,1.348739,False
3,31603,Insurance,IDN,0.022178,NaN,0.461366,False
4,11333,Capital Goods,CHN,0.071349,NaN,0.555100,False


# 2. What is wrong with the data?

Before we change anything, we count the different problems, since they need different treatment:

- Missing values (`NaN`): we just don't know the value.
- `inf` and `-inf`: a division by zero. A margin is profit divided by revenue, so this probably means that revenue is zero.
- Negative asset turnover: revenue divided by assets should not be negative.

In [3]:
rows = []
for ratio in RATIOS:
    v = data[ratio]
    finite = v[np.isfinite(v)]
    rows.append({
        "ratio": ratio,
        "missing": v.isna().sum(),
        "missing %": 100 * v.isna().mean(),
        "+inf": (v == np.inf).sum(),
        "-inf": (v == -np.inf).sum(),
        "zero": (v == 0).sum(),
        "negative": (finite < 0).sum(),
        "min": finite.min(),
        "median": finite.median(),
        "max": finite.max(),
    })
display(pd.DataFrame(rows).set_index("ratio").round(3))

,missing,missing %,+inf,-inf,zero,negative,min,median,max
ratio,,,,,,,,,
net_profit_margin,3038,11.199,182,1820,2,7521,-145480.000,0.038,3.507696e+07
ebitda_margin,16895,62.279,66,747,1,2978,-2001935.190,0.103,3.504681e+07
asset_turnover,3259,12.013,5,0,1941,117,-1788.667,0.462,1.259940e+06


There is a lot going on here. Net margin has about 2,000 `inf` values and EBITDA about 800. The extreme values are also huge compared to the median: net margin goes up to about 35 million, while its median is 0.04. EBITDA is missing for 62% of the companies, much more than the other two ratios.

We looked at the 117 negative asset turnovers separately to see if they are just values with a wrong sign. They don't look like it: they are much smaller than normal values, and 65% of them are Financial Services companies. This would fit companies with negative revenue, for example from investment losses, but we can't prove that from this file. Either way the number is not useful, so we treat them as missing.

### Does `inf` really mean zero revenue?

If an `inf` margin means zero revenue, these companies should also have an asset turnover of 0, since revenue is on top of that ratio. So we can test this instead of just assuming it.

In [4]:
pd.crosstab(np.isinf(data["net_profit_margin"]), data["asset_turnover"] == 0,
            rownames=["net margin is inf"], colnames=["asset turnover is 0"])

asset turnover is 0,False,True
net margin is inf,,
False,25123,3
True,64,1938


This fits. Of the 2,002 companies with an `inf` net margin, 1,938 also have an asset turnover of 0, and only 3 companies have a turnover of 0 without an `inf` margin. So `inf` really does seem to mean zero revenue.

### Checking the country column

Country is not a number, so the table above does not work for it. There are no `inf` values or negative numbers to count. Instead we check what can go wrong with a category: missing values, broken codes and countries with almost no companies.

In [ ]:
counts = data["country_code"].value_counts()
rare = counts[counts < MIN_COMPANIES]

print(f"missing: {data['country_code'].isna().sum()} ({100 * data['country_code'].isna().mean():.1f}%)")
print("missing exactly where there is no financial row:", (data["country_code"].isna() == data["no_financial_row"]).all())
print("codes that are not three capital letters:", (~counts.index.str.fullmatch("[A-Z]{3}")).sum())
print("countries:", len(counts))
print(f"countries with fewer than {MIN_COMPANIES} companies: {len(rare)}, with {rare.sum()} companies in total")
display(counts.head(8).to_frame("companies").assign(**{"share %": lambda t: (100 * t["companies"] / len(data)).round(1)}))

Country looks clean. It is only missing for the 8.1% of companies with no financial row, and all the codes look fine. USA and China alone make up about a third of the data. But 53 of the 98 countries have fewer than 20 companies, so we need to do something about those.

# 3. Cleaning the data

Based on what we found, we use these rules:

- A company with an `inf` margin or an asset turnover of 0 gets a **zero-revenue flag**. This is real information, so we want to keep it.
- The `inf` margin itself becomes missing, since there is no usable number.
- Negative or `inf` asset turnovers become missing.
- Missing values stay missing for now. Section 4 looks at how to fill them for a model.
- Countries with fewer than 20 companies go into a group called `other`, because we can't say much about a country with so few companies.
- A missing country stays missing here. For a model it becomes `unknown`.

`financial_data.ipynb` replaces `inf` with a median instead, but then a company without revenue looks like an average company. The flag keeps that information.

The `*_clean` columns are in the original units and still have gaps, so the statistics in section 6 only use values that were really observed. `country_group` is the same idea for country.

In [5]:
data["zero_revenue"] = np.isinf(data["net_profit_margin"]) | (data["asset_turnover"] == 0)

for ratio in RATIOS:
    v = data[ratio].replace([np.inf, -np.inf], np.nan)
    if ratio == "asset_turnover":
        v = v.mask(v < 0)
    data[ratio + "_clean"] = v

data["country_group"] = data["country_code"].mask(data["country_code"].map(counts) < MIN_COMPANIES, "other")

print("zero-revenue companies:", data["zero_revenue"].sum())
print(data[[r + "_clean" for r in RATIOS]].isna().mean().round(3))

zero-revenue companies: 2005
net_profit_margin_clean    0.186
ebitda_margin_clean        0.653
asset_turnover_clean       0.125
dtype: float64


2,005 companies get the zero-revenue flag. After cleaning, EBITDA is missing for 65% of the companies, net margin for 19% and asset turnover for 13%.

# 4. Missing data

### Is it random?

Before we fill any gaps, we want to know why they exist. If values are missing at random, filling them in is harmless. But if some industries have far more gaps than others, then a gap tells us something about the company, and we should keep that information.

We check this with a **chi-square test of independence**. If the gaps had nothing to do with industry, every industry would have about the same share of them.

With about 27,000 companies even a tiny difference gives a very small p-value, so the p-value alone doesn't tell us much. That is why we also report **Cramér's V**, which shows how strong the link is, from 0 (no link) to 1 (perfect link). As a rough guide, 0.1 is small, 0.3 is medium and 0.5 is large.

In [6]:
flags = {
    "no financial row": data["no_financial_row"],
    "EBITDA missing": data["ebitda_margin_clean"].isna() & ~data["no_financial_row"],
    "zero revenue": data["zero_revenue"],
}

rows = []
for name, flag in flags.items():
    table = pd.crosstab(data["industry"], flag)
    chi2, p, dof, _ = stats.chi2_contingency(table)
    rows.append({"flag": name, "share %": 100 * flag.mean(), "chi2": chi2, "p": p,
                 "Cramer's V": np.sqrt(chi2 / len(data))})
display(pd.DataFrame(rows).set_index("flag").round(3).style.format({"p": "{:.1e}"}, precision=3))

,share %,chi2,p,Cramer's V
flag,,,,
no financial row,8.132,187.419,0.0e+00,0.083
EBITDA missing,57.144,1377.563,0.0e+00,0.225
zero revenue,7.391,2855.171,0.0e+00,0.324


In [ ]:
shares = pd.DataFrame({name: flag.groupby(data["industry"]).mean() for name, flag in flags.items()})
shares = shares.sort_values("EBITDA missing")

plt.figure(figsize=(8, 8))
sns.heatmap(shares, annot=True, fmt=".0%", cmap="Blues", vmin=0, vmax=1)
plt.title("Share of companies with each data problem per industry")
plt.xlabel("Data problem")
plt.ylabel("Industry")
plt.tight_layout()
plt.show()

The gaps are clearly not random. All p-values are close to 0, and the link is strongest for zero revenue (V = 0.32), then EBITDA missing (V = 0.23), while no financial row is small (V = 0.08).

In the plot we can see that zero revenue is most common in Materials (24%) and Pharma (20%). That makes sense to us, since mining and biotech companies often don't have any revenue yet. EBITDA is missing most often for Semiconductors (75%) and least often for REITs (33%).

So a gap is a signal here, and we want to keep it as a flag instead of just filling it in.

Country does not need its own row in the table. It is missing for exactly the companies in `no financial row`, so that row is already its test.

### What is the best way to fill the gaps?

A model needs a number in every cell, so we have to fill the gaps somehow. We can't drop those companies, because the final model has to predict every company. Dropping `ebitda_margin` would lose the information about which companies are missing it. And filling with the industry median is not allowed: industry is what we want to predict, so that would leak the answer.

That leaves the country median, which is used in `financial_data.ipynb`, and the global median. To compare them, we hide 20% of the known values, fill them in with each rule and measure how far off we are. We do this in signed-log units (explained in section 5), so that a few giant values don't decide the result.

In [8]:
def signed_log(v):
    return np.sign(v) * np.log1p(np.abs(v))


rng = np.random.default_rng(0)
rows = []
for ratio in RATIOS:
    known = data[[ratio + "_clean", "country_code"]].dropna()
    hidden = rng.random(len(known)) < 0.2
    fit, test = known[~hidden], known[hidden]

    global_median = fit[ratio + "_clean"].median()
    grouped = fit.groupby("country_code")[ratio + "_clean"]
    country_median = grouped.median().where(grouped.count() >= 10)   # same rule as Daniel: at least 10 values
    by_country = test["country_code"].map(country_median).fillna(global_median)

    truth = signed_log(test[ratio + "_clean"])
    rows.append({
        "ratio": ratio,
        "error global median": (truth - signed_log(global_median)).abs().mean(),
        "error country median": (truth - signed_log(by_country)).abs().mean(),
    })
display(pd.DataFrame(rows).set_index("ratio").round(4))

,error global median,error country median
ratio,,
net_profit_margin,0.3844,0.3817
ebitda_margin,0.6095,0.6038
asset_turnover,0.2870,0.2699


The country median is a bit better for all three ratios, but the difference is really small. For asset turnover, which has the biggest difference, the error goes from 0.287 to 0.270. So we stay with the global median, since it is simpler.

Filling gaps always makes the data look more certain than it is, because lots of companies get the same value. For EBITDA that is most of the rows. This is why every ratio also gets a **missing flag**, so the model can learn that the number was made up.

Country needs no filling rule. A company without a country just gets its own `unknown` category, which does the same job as the missing flag.

# 5. Outliers

### How bad is it?

A few huge values can completely destroy the mean and the standard deviation, while the median stays sensible. To count the outliers we use the **Tukey fence**: everything more than 3 × IQR beyond the quartiles counts as far out. The IQR is based on quartiles, so the outliers can't distort the rule that is supposed to find them. A rule based on the mean and standard deviation would have exactly this problem.

In [9]:
rows = []
for ratio in RATIOS:
    v = data[ratio + "_clean"].dropna()
    q1, q3 = v.quantile([0.25, 0.75])
    low, high = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
    rows.append({"ratio": ratio, "mean": v.mean(), "std": v.std(), "median": v.median(),
                 "lower fence": low, "upper fence": high, "far out %": 100 * ((v < low) | (v > high)).mean()})
display(pd.DataFrame(rows).set_index("ratio").round(3))

,mean,std,median,lower fence,upper fence,far out %
ratio,,,,,,
net_profit_margin,1805.841,238732.848,0.038,-0.702,0.748,15.284
ebitda_margin,3323.686,361890.595,0.103,-1.178,1.311,15.637
asset_turnover,54.117,8176.109,0.465,-2.037,3.022,1.735


The mean net margin is about 1,806 while the median is 0.04, so the mean tells us nothing here. About 15% of the margins are far out, but only 1.7% of the asset turnovers.

### Are they errors?

Before we remove anything, we look at the most extreme companies. If they are real, we would be deleting real information.

In [10]:
columns = ["industry", "country_code", "net_profit_margin_clean", "ebitda_margin_clean", "asset_turnover_clean"]
display(data.nlargest(5, "net_profit_margin_clean")[columns])
display(data.nsmallest(5, "net_profit_margin_clean")[columns])

,industry,country_code,net_profit_margin_clean,ebitda_margin_clean,asset_turnover_clean
22187,Materials,AUS,3.507696e+07,3.504681e+07,NaN
23760,Financial Services,CAN,5.335038e+06,-1.125820e+06,NaN
6544,Media & Entertainment,SWE,4.329286e+04,4.325621e+04,0.000006
20539,Materials,AUS,1.325072e+04,-5.512984e+03,0.000019
6819,"Pharmaceuticals, Biotechnology & Life Sciences",USA,8.437860e+03,8.516750e+03,0.001106


,industry,country_code,net_profit_margin_clean,ebitda_margin_clean,asset_turnover_clean
21759,Financial Services,USA,-145480.000000,NaN,0.000052
13153,Real Estate Management & Development,USA,-32113.666667,NaN,0.039735
7013,"Pharmaceuticals, Biotechnology & Life Sciences",USA,-31028.750000,-28513.500000,0.000128
15786,Energy,POL,-26810.928571,NaN,0.000165
18092,Consumer Services,SGP,-23813.333333,-19166.666667,0.000513


Most of these companies have an asset turnover close to 0, so their revenue is tiny compared to their assets. A margin divides by revenue, so a tiny revenue gives a giant number. These values are probably real, they are just not very useful: the company is not really that profitable, the ratio just blows up.

So we **shrink them instead of deleting them**, in two steps:

1. **Signed log**, `sign(v) * log(1 + |v|)`. This compresses huge values and keeps small ones almost the same. Unlike a normal log it also works for negative margins.
2. **Clipping** at the 1st and 99th percentile. Everything beyond that is set to the limit, so the company stays in the data but can't dominate it anymore.

In [ ]:
plt.figure(figsize=(13, 6))

for i, ratio in enumerate(RATIOS):
    before = data[ratio + "_clean"].dropna()
    after = signed_log(before)
    after = after.clip(*after.quantile([0.01, 0.99]))
    data[ratio + "_scaled"] = after

    # Top row: cleaned values, bottom row: after the signed log and clipping
    plt.subplot(2, 3, i + 1)
    plt.hist(before, bins=60, log=True)
    plt.title(ratio + " (cleaned)")
    plt.xlabel("Value")
    plt.ylabel("Count (log scale)")

    plt.subplot(2, 3, i + 4)
    plt.hist(after, bins=60)
    plt.title(ratio + " (signed log + clipped)")
    plt.xlabel("Value")
    plt.ylabel("Count")

plt.tight_layout()
plt.show()

scaled = [r + "_scaled" for r in RATIOS]
display(data[scaled].agg(["mean", "std", "median", "min", "max"]).round(3))

In the top row almost all companies are in one bar, because a few extreme companies stretch the whole axis. After the signed log and the clipping we can actually see the shape of the data. The margins sit tightly around 0 with a long tail of companies making losses, while asset turnover is much more spread out.

The small peaks at the ends of the bottom plots are the clipped values, and the tall bar at 0 for asset turnover is mostly the zero-revenue companies. No company was removed, only the scale changed. The `mean` and `std` in the table also look sensible now.

### Outliers in country

A category has no extreme values, so the Tukey fence, the signed log and the clipping don't work here. The closest thing to an outlier is a country with very few companies. A model would get a whole column for that country, but almost nothing to learn from it.

We use the same idea as for the ratios: shrink, don't delete. These companies stay in the data, but in the `other` group. The plot shows the countries before and after.

In [ ]:
groups = data["country_group"].value_counts()

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.bar(range(len(counts)), counts.values)
plt.axhline(MIN_COMPANIES, color="black", linestyle="--", label=f"{MIN_COMPANIES} companies")
plt.yscale("log")
plt.title(f"country_code ({len(counts)} countries)")
plt.xlabel("Countries, largest first")
plt.ylabel("Companies (log scale)")
plt.legend()

plt.subplot(1, 2, 2)
plt.bar(range(len(groups)), groups.values)
plt.yscale("log")
plt.title(f"country_group ({len(groups)} groups)")
plt.xlabel("Countries, largest first")
plt.ylabel("Companies (log scale)")

plt.tight_layout()
plt.show()

print("size of the other group:", groups["other"])

The 53 smallest countries end up in `other` together, with 220 companies in total. That leaves 46 groups, which is the same number of country columns a model would get.

# 6. Do the financial distributions differ between industries?

### The typical company in each industry

We use medians and quartiles here, because they are not really affected by outliers. All values are real observed values, so the medians we filled in for section 4 can't create fake differences.

In [12]:
summary = data.groupby("industry")[[r + "_clean" for r in RATIOS]].median()
summary.columns = RATIOS
summary.insert(0, "companies", data["industry"].value_counts())
display(summary.sort_values("asset_turnover").round(3))

,companies,net_profit_margin,ebitda_margin,asset_turnover
industry,,,,
Banks,947,0.246,0.249,0.037
Equity Real Estate Investment Trusts (REITs),422,0.251,0.570,0.083
Financial Services,1884,0.258,0.347,0.091
Real Estate Management & Development,898,0.075,0.220,0.130
Insurance,287,0.079,0.130,0.239
"Pharmaceuticals, Biotechnology & Life Sciences",1922,-0.185,-1.028,0.241
Utilities,569,0.083,0.301,0.269
Energy,1316,0.040,0.155,0.382
Materials,3917,0.025,0.077,0.401


The differences look very reasonable. Banks, REITs and Financial Services have the lowest asset turnover (0.04 to 0.09), since they have a lot of assets compared to their revenue. Retailers have the highest, with Consumer Staples Retail at 1.69. Pharma stands out with a median net margin of -0.19 and a median EBITDA margin of -1.03, so a typical company there is losing money.

In [ ]:
def plot_industry_distribution(ratio):
    # Sort the industries by their median, so the plot is easier to read
    order = data.groupby("industry")[ratio + "_scaled"].median().sort_values().index

    plt.figure(figsize=(10, 7))
    sns.boxplot(data=data, x=ratio + "_scaled", y="industry", order=order, showfliers=False, color="lightblue")
    plt.title(ratio + " by industry")
    plt.xlabel(ratio + " (signed log + clipped)")
    plt.ylabel("Industry")
    plt.tight_layout()
    plt.show()


for ratio in RATIOS:
    plot_industry_distribution(ratio)

We use the scaled values from section 5 for these plots, since the boxes would be impossible to read otherwise. The box shows the middle 50% of the companies and the line in the box is the median.

Asset turnover shows the clearest differences, with the medians spread nicely from Banks to retail. For the margins most boxes sit close to 0. Pharma is the main exception on the low side, and Financial Services and REITs on the high side.

### Countries in each industry

A category has no median, so we describe the typical company differently: which country is most common in each industry, and which countries appear more often than we would expect.

For this we calculate how over-represented an industry is in a country: its share among the companies of that country, divided by its share among all companies. 1 means normal, 2 means the industry is twice as common in that country. We only show the 8 largest countries, because smaller countries have too few companies per industry.

In [ ]:
known = data.dropna(subset=["country_group"])
country_mix = pd.crosstab(known["industry"], known["country_group"], normalize="index")
display(pd.DataFrame({"companies": known["industry"].value_counts(), "most common country": country_mix.idxmax(axis=1),
                      "share %": (100 * country_mix.max(axis=1)).round(1)}).sort_values("share %", ascending=False))

In [ ]:
largest = counts.index[:8]
overall = known["industry"].value_counts(normalize=True)
lift = pd.crosstab(known["industry"], known["country_group"], normalize="columns")[largest].div(overall, axis=0)
lift = lift.sort_values(largest[0])

plt.figure(figsize=(9, 8))
sns.heatmap(lift, annot=True, fmt=".1f", cmap="coolwarm", center=1, vmin=0, vmax=2)
plt.title("Over-representation of each industry per country (1 = as expected)")
plt.xlabel("Country")
plt.ylabel("Industry")
plt.tight_layout()
plt.show()

There are some clear patterns. Taiwan has 5 times the usual share of Semiconductor companies and 4.6 times the share of Technology Hardware. Materials is common in Canada and Australia (3.1 and 2.7) and Financial Services in the UK (3.6). Banks are the most concentrated industry, 57% of them are in the USA. The color scale stops at 2, so everything above that has the same color.

### Are the differences significant?

We use the **Kruskal-Wallis test**. The null hypothesis is that all 25 industries come from the same distribution. It works like ANOVA but on ranks. ANOVA compares means and assumes roughly normal data, and section 5 showed that neither makes sense here. With ranks, it doesn't matter how extreme an outlier is, only that it is the largest value.

With 27,000 companies almost every difference gets a tiny p-value, so we also look at the effect size. **Eta-squared** is the share of the variation in the ranks that is explained by the industry. As a rough guide, 0.01 is small, 0.06 is medium and 0.14 is large.

In [ ]:
rows = []
for ratio in RATIOS:
    groups = [g.dropna().to_numpy() for _, g in data.groupby("industry")[ratio + "_clean"]]
    h, p = stats.kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    rows.append({"ratio": ratio, "companies": n, "H": h, "p": p, "eta squared": (h - k + 1) / (n - k)})
display(pd.DataFrame(rows).set_index("ratio").style.format({"p": "{:.1e}"}, precision=4))

All three ratios differ between industries. Asset turnover is clearly the strongest (0.25), then EBITDA margin (0.19) and net margin (0.13). The EBITDA number only uses the 9,420 companies that have a real value.

For country we can't use Kruskal-Wallis, because it needs values that can be ranked and countries have no order. So we use the chi-square test and Cramér's V from section 4 again.

Two things are different from section 4:

- The normal V gets pushed up if a table has many categories and few companies per cell, so we use the **bias-corrected** version. For the flags in section 4 this correction is too small to matter.
- With many categories V is not on the same scale as for the flags, and it can't be compared with eta-squared at all. So we add a second row, USA against the rest. It has two columns like the flags in section 4, so those two can be compared.

Chi-square needs an expected count of about 5 per cell. The last column shows how many cells are below that.

In [ ]:
def cramers_v(table):
    # Bias-corrected Cramer's V (Bergsma 2013)
    chi2 = stats.chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    r, c = table.shape
    phi2 = max(0, chi2 / n - (r - 1) * (c - 1) / (n - 1))
    r_corrected, c_corrected = r - (r - 1) ** 2 / (n - 1), c - (c - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / (min(r_corrected, c_corrected) - 1))


tables = {
    f"country ({known['country_group'].nunique()} groups)": pd.crosstab(known["industry"], known["country_group"]),
    f"{largest[0]} vs the rest": pd.crosstab(known["industry"], known["country_group"] == largest[0]),
}

rows = []
for name, table in tables.items():
    chi2, p, dof, expected = stats.chi2_contingency(table)
    rows.append({"variable": name, "chi2": chi2, "dof": dof, "p": p, "Cramer's V": cramers_v(table),
                 "cells with expected < 5 %": 100 * (expected < 5).mean()})
display(pd.DataFrame(rows).set_index("variable").style.format({"p": "{:.1e}"}, precision=3))

Country is clearly linked to industry. With all 46 groups V is 0.15, and USA against the rest gives 0.30, which is about as strong as the zero-revenue flag. 49% of the cells in the first row are below an expected count of 5, so we trust the second row more.

### Can the financials separate the industries our classifier confuses?

Kruskal-Wallis only tells us that some industries differ. For classification it matters more if the pairs our models mix up are different. The pairs below are some of the most frequent confusions of the ensemble.

We use the **Mann-Whitney U test** for each pair. Since we run a lot of tests, a few would look significant just by luck. The **Bonferroni correction** handles this by multiplying each p-value by the number of tests.

As effect size we use the **AUC**: the probability that a random company from the first industry has a higher value than one from the second. 0.5 means no separation, 1.0 or 0.0 means perfect separation. We get it directly from the test statistic with `AUC = U / (n1 * n2)`.

In [15]:
CONFUSED_PAIRS = [
    ("Capital Goods", "Materials"),
    ("Technology Hardware & Equipment", "Capital Goods"),
    ("Technology Hardware & Equipment", "Software & Services"),
    ("Health Care Equipment & Services", "Pharmaceuticals, Biotechnology & Life Sciences"),
    ("Energy", "Materials"),
    ("Commercial & Professional Services", "Capital Goods"),
    ("Commercial & Professional Services", "Software & Services"),
    ("Semiconductors & Semiconductor Equipment", "Technology Hardware & Equipment"),
    ("Consumer Discretionary Distribution & Retail", "Consumer Durables & Apparel"),
    ("Automobiles & Components", "Capital Goods"),
    ("Software & Services", "Financial Services"),
]

rows = []
for a, b in CONFUSED_PAIRS:
    for ratio in RATIOS:
        values_a = data.loc[data["industry"] == a, ratio + "_clean"].dropna()
        values_b = data.loc[data["industry"] == b, ratio + "_clean"].dropna()
        u, p = stats.mannwhitneyu(values_a, values_b)
        rows.append({"pair": f"{a} vs {b}", "ratio": ratio, "median first": values_a.median(),
                     "median second": values_b.median(), "AUC": u / (len(values_a) * len(values_b)), "p": p})

pairs = pd.DataFrame(rows)
pairs["p corrected"] = (pairs["p"] * len(pairs)).clip(upper=1)
pairs["significant"] = pairs["p corrected"] < 0.05
print(f"{pairs['significant'].sum()} of {len(pairs)} comparisons significant after correction")
display(pairs.pivot(index="pair", columns="ratio", values="AUC").loc[[f"{a} vs {b}" for a, b in CONFUSED_PAIRS], RATIOS].round(2))
display(pairs.assign(separation=(pairs["AUC"] - 0.5).abs()).nlargest(10, "separation")
        .drop(columns=["separation", "significant"]).style.format({"p": "{:.1e}", "p corrected": "{:.1e}"}, precision=3))

21 of 33 comparisons significant after correction


ratio,net_profit_margin,ebitda_margin,asset_turnover
pair,,,
Capital Goods vs Materials,0.56,0.57,0.65
Technology Hardware & Equipment vs Capital Goods,0.45,0.39,0.52
Technology Hardware & Equipment vs Software & Services,0.56,0.51,0.52
"Health Care Equipment & Services vs Pharmaceuticals, Biotechnology & Life Sciences",0.57,0.64,0.69
Energy vs Materials,0.55,0.65,0.55
Commercial & Professional Services vs Capital Goods,0.49,0.50,0.53
Commercial & Professional Services vs Software & Services,0.59,0.61,0.53
Semiconductors & Semiconductor Equipment vs Technology Hardware & Equipment,0.55,0.55,0.38
Consumer Discretionary Distribution & Retail vs Consumer Durables & Apparel,0.46,0.48,0.64


,pair,ratio,median first,median second,AUC,p,p corrected
32,Software & Services vs Financial Services,asset_turnover,0.613,0.091,0.814,2.3e-181,7.5e-180
30,Software & Services vs Financial Services,net_profit_margin,-0.002,0.258,0.240,7.6e-119,2.5e-117
31,Software & Services vs Financial Services,ebitda_margin,0.016,0.347,0.242,3.2e-65,1.1e-63
11,"Health Care Equipment & Services vs Pharmaceuticals, Biotechnology & Life Sciences",asset_turnover,0.542,0.241,0.694,7.0e-59,2.3e-57
2,Capital Goods vs Materials,asset_turnover,0.612,0.401,0.651,3.5e-95,1.1e-93
13,Energy vs Materials,ebitda_margin,0.155,0.077,0.646,1.8e-17,5.9e-16
10,"Health Care Equipment & Services vs Pharmaceuticals, Biotechnology & Life Sciences",ebitda_margin,-0.014,-1.028,0.642,3.0e-17,9.8e-16
26,Consumer Discretionary Distribution & Retail vs Consumer Durables & Apparel,asset_turnover,1.076,0.749,0.640,2.3e-20,7.6e-19
23,Semiconductors & Semiconductor Equipment vs Technology Hardware & Equipment,asset_turnover,0.491,0.630,0.383,3.6e-16,1.2e-14
19,Commercial & Professional Services vs Software & Services,ebitda_margin,0.098,0.016,0.610,5.8e-09,1.9e-07


21 of the 33 comparisons are significant. The best one by far is Software & Services vs Financial Services with asset turnover (AUC 0.81). Health Care vs Pharma (0.69) and Capital Goods vs Materials (0.65) also separate okay with asset turnover. Others barely separate at all, for example Commercial & Professional Services vs Capital Goods (0.49 to 0.53).

Even an AUC of 0.65 still means a lot of overlap, so the financials alone can't tell these industries apart. They can only give the model a small hint.

We do the same for country. Mann-Whitney also needs values that can be ranked, so each pair gets a chi-square test and the bias-corrected V instead, with the Bonferroni correction for 11 tests.

V can't be compared with the AUC above, and many cells are small here, so this is mostly useful as a ranking of the pairs. The last columns show the country whose share is furthest apart between the two industries.

In [ ]:
rows = []
for a, b in CONFUSED_PAIRS:
    subset = known[known["industry"].isin([a, b])]
    table = pd.crosstab(subset["country_group"], subset["industry"])
    p = stats.chi2_contingency(table)[1]
    shares_by_country = table / table.sum()
    gap = (shares_by_country[a] - shares_by_country[b]).abs().idxmax()
    rows.append({"pair": f"{a} vs {b}", "Cramer's V": cramers_v(table), "p corrected": min(p * len(CONFUSED_PAIRS), 1),
                 "biggest gap": gap, "share first %": 100 * shares_by_country.loc[gap, a],
                 "share second %": 100 * shares_by_country.loc[gap, b]})

country_pairs = pd.DataFrame(rows).set_index("pair").sort_values("Cramer's V", ascending=False)
print(f"{(country_pairs['p corrected'] < 0.05).sum()} of {len(country_pairs)} pairs significant after correction")
display(country_pairs.style.format({"p corrected": "{:.1e}"}, precision=2))

All 11 pairs are significant. The strongest are Capital Goods vs Materials, where Canada is 2% against 26%, and Technology Hardware vs Software & Services, where Taiwan is 23% against 2%. The weakest is Automobiles vs Capital Goods.

# 7. Conclusions

Still to fill in after running the notebook:

1. **Zero revenue:** does the check in section 2 support the idea that `inf` margins mean zero revenue?
2. **Missing data:** is it random? Which industries stand out, and does that make sense for their business?
3. **Filling gaps:** is the country median enough better than the global median to justify the extra rule?
4. **Outliers:** are the extreme companies errors or real values? What changed in the `mean` and `std`?
5. **Differences:** which ratio separates the industries best, going by effect size and not by p-value?
6. **For the classifier:** which confused pairs do the financials separate well, and which not at all? How does this compare with what the financials add to the ensemble in `02_final_ensembles.ipynb`?
7. **Country:** how strong is the link with industry, and which confused pairs does it separate? Is that worth about 46 extra columns, given what country adds to the ensemble?